In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from tqdm import tqdm
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)

from ultralytics import YOLO
from cv_utils import *
from cv_pipeline import *

In [3]:
# YOLO model path
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
pitch_segment_model = smp.Unet("resnet34", encoder_weights="imagenet", activation=None, classes=7)
pitch_segment_model.load_state_dict(torch.load(os.path.join(model_path, "best res_unet_512.pth")))

player_detect_model = YOLO(os.path.join(model_path, "yolov8n_4th_train.pt"))

# General data path
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')

#### Crop images to football pitch

In [4]:
all_cropped_destination = []
for vid_name in ['Liv-Ars-train']:
    print(vid_name)
    image_root_path = os.path.join(data_path, 'images/' + vid_name)
    destination_path = os.path.join(data_path, 'images/cropped_' + vid_name)
    if not os.path.exists(destination_path):
        os.mkdir(destination_path)
    for image_name in tqdm(os.listdir(image_root_path)):
        image_path = os.path.join(image_root_path, image_name)
        image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
        orig_w, orig_h = image.shape[:2]

        # Crop the image to only the pitch
        predicted_mask, colored_mask = pitch_segment(image, pitch_segment_model)
        pitch_area = cv2.inRange(predicted_mask, 41, 255)
        _, rect, _ = detect_largest_contour(pitch_area)

        startX, startY, w, h = rect
        endX, endY = startX + w, startY + h
        cropped_image = cv2.resize(image[startY:endY, startX:endX], (orig_h, orig_w))

        # Write image to destination
        destination_image_path = os.path.join(destination_path, image_name)
        cv2.imwrite(destination_image_path, cropped_image)
    print(destination_path)
    all_cropped_destination.append(destination_path)

Liv-Ars-train


  4%|▍         | 7/164 [00:06<02:21,  1.11it/s]


KeyboardInterrupt: 

#### Generate COCO dataset

In [6]:
# Generate one or multiple COCO datasets
destination_path_list = []
for folder_path in all_cropped_destination:
    vid_name = os.path.basename(folder_path)
    print(vid_name)
    print(os.path.join(data_path, 'images' + '/' + vid_name))
    dest_coco_path = export_coco_dataset_from_prediction(data_path, vid_name, 
                                        model_path=model_path, model_name="yolov8n_3rd_train.pt")
    
    destination_path_list.append(dest_coco_path)

cropped_Liv-Ars-train
/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_Liv-Ars-train


  0%|          | 0/164 [00:00<?, ?it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_Liv-Ars-train/000000000021.png: 1152x2048 21 persons, 8.1ms
Speed: 6.4ms preprocess, 8.1ms inference, 0.6ms postprocess per image at shape (1, 3, 1152, 2048)
  1%|          | 1/164 [00:00<00:29,  5.53it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_Liv-Ars-train/000000000022.png: 1152x2048 20 persons, 7.8ms
Speed: 6.4ms preprocess, 7.8ms inference, 0.8ms postprocess per image at shape (1, 3, 1152, 2048)
  1%|          | 2/164 [00:00<00:21,  7.37it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_Liv-Ars-train/000000000023.png: 1152x2048 20 persons, 7.7ms
Speed: 6.6ms preprocess, 7.7ms inference, 1.2ms postprocess per image at shape (1, 3, 1152, 2048)
  2%|▏         | 3/164 [00:00<00:19,  8.07it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Fo

#### Merge multiple COCO datasets

In [9]:
coco_dataset_path_1 = os.path.join(data_path, 'coco_datasets/cropped_Liv-Ars-train fixed')
coco_dataset_path_2 = os.path.join(data_path, 'coco_datasets/cropped_real_synth_train_fixed')

dest_path=os.path.join(data_path, 'coco_datasets')

# Merge generated COCO dataset to one dataset for model training
merge_dest_path = merge_coco_dataset(coco_dataset_path_1, coco_dataset_path_2,
                   dest_path=os.path.join(data_path, 'coco_datasets'))

#### Convert COCO dataset to YOLO format

In [10]:
# Load COCO dataset
dataset_name = 'cropped_all_match'
coco_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
test_dataset_path = os.path.join(data_path, 'coco_datasets/' + 'real_test_fixed')
# Convert COCO to YOLO
coco2yolo(coco_dataset_path, test_dataset_path=test_dataset_path)

100%|██████████| 300/300 [00:24<00:00, 12.19it/s]


#### Augment YOLO dataset

In [11]:
dataset_name = 'cropped_Liv-Ars-train fixed_yolov8'
yolo_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
yolo_metadata_path = os.path.join(yolo_dataset_path, 'data.yaml')

augment_yolo(yolo_metadata_path, yolo_dataset_path)

/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/coco_datasets/cropped_Liv-Ars-train fixed_yolov8/train/images


  0%|          | 0/1179 [00:00<?, ?it/s]

100%|██████████| 1179/1179 [08:35<00:00,  2.29it/s]

9563 9563
